# Models, experiments, training state, and metrics

This standalone lesson requires Python 3.10-3.13 and the matching installed DRYML version with its `sklearn` extra (`dryml[sklearn]`); that extra is not installed on Python 3.14. It uses only tiny fixed in-memory NumPy arrays, runs offline, and keeps all Store state in a temporary directory.

The maintained sklearn path has four roles: `RegressionModel` adapts an estimator as a DRYML `Model`, `BasicTraining` fits it from a supervised dataset, `Experiment` groups model, trainer, and data, and `TrainState` records lifecycle progress. Metrics are evaluated explicitly against a model and dataset.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np
from sklearn.linear_model import LinearRegression

from dryml.artifacts import CachedDataset
from dryml.core2 import Repo, TensorSpec
from dryml.core2.store import DirStore
from dryml.data import ArrayDataset, Map
from dryml.metrics import mean_squared_error
from dryml.models import Experiment, TrainState
from dryml.models.sklearn import BasicTraining, RegressionModel

## Fixed supervised data

`ArrayDataset((x, y))` yields aligned feature-target pairs. These three local examples define a deterministic linear relation and make the later step count visible.

In [ ]:
x_train = np.array(
    [[0.0, 10.0], [1.0, 11.0], [2.0, 12.0]],
    dtype=np.float32,
)
y_train = np.array([1.0, 2.0, 3.0], dtype=np.float32)
train_data = ArrayDataset((x_train, y_train))

assert len(train_data) == 3
first_x, first_y = train_data.peek()
np.testing.assert_array_equal(first_x, np.array([0.0, 10.0], dtype=np.float32))
assert first_y == np.float32(1.0)

## Experiment lifecycle

A new experiment starts in `TrainState.initial` (`None`). `Experiment.train()` sets the phase to `training` before invoking `BasicTraining`; successful sklearn fitting advances one epoch and one step per collated example, then leaves the phase `trained`. The fitted estimator is mutable object state, not a new model definition.

In [ ]:
model = RegressionModel(LinearRegression)
experiment = Experiment(model, BasicTraining(), train_data=train_data)

assert experiment.state.phase is TrainState.initial
assert experiment.state.is_initial
assert TrainState.training == 'training'
assert TrainState.trained == 'trained'

training_result = experiment.train()

assert training_result is model.estimator
assert model.obj is model.estimator
assert experiment.state.phase == TrainState.trained
assert experiment.state.is_trained
assert experiment.state.epoch == 1
assert experiment.state.step == 3

## A model is a method

Calling the wrapper delegates to estimator prediction. Because `Model` is also a DRYML `Method`, `Map` can apply it to dataset elements and infer an output spec. Here each source element is explicitly a batch of two feature rows, matching sklearn's two-dimensional prediction input.

In [ ]:
single_prediction = model(np.array([[3.0, 13.0]], dtype=np.float32))
np.testing.assert_allclose(single_prediction, np.array([4.0]), atol=1e-6)

input_spec = TensorSpec('float32', shape=(2,), backend='numpy')
assert model.infer_output_spec(input_spec) == TensorSpec(
    'float32',
    shape=(),
    backend='numpy',
)

feature_batches = np.array(
    [
        [[0.0, 10.0], [3.0, 13.0]],
        [[1.0, 11.0], [2.0, 12.0]],
    ],
    dtype=np.float32,
)
batch_spec = TensorSpec('float32', shape=(2,), batch=2, backend='numpy')
prediction_batches = Map(ArrayDataset(feature_batches, spec=batch_spec), model)

assert prediction_batches.spec == TensorSpec(
    'float32',
    shape=(),
    batch=2,
    backend='numpy',
)
np.testing.assert_allclose(
    list(prediction_batches),
    np.array([[1.0, 4.0], [2.0, 3.0]], dtype=np.float32),
    atol=1e-6,
)

## Explicit metric evaluation

Metrics are ordinary explicit computations: supply the trained model and evaluation dataset, and retain or publish the scalar deliberately. The fixed training relation has deterministic zero mean squared error.

In [ ]:
metric_value = mean_squared_error(model, train_data, batch_size=3)

assert isinstance(metric_value, float)
np.testing.assert_allclose(metric_value, 0.0, atol=1e-12)

## Handled training failure

Current training validates that an experiment has training data. This failure case uses a separately configured estimator definition. `Experiment.train()` records `failed` before re-raising the error; handling it does not advance epoch or step and does not require displaying a traceback.

In [ ]:
failed_experiment = Experiment(
    RegressionModel(LinearRegression, fit_intercept=False),
    BasicTraining(),
)

try:
    failed_experiment.train()
except ValueError as error:
    assert str(error) == 'Experiment has no train_data.'
else:
    raise AssertionError('missing training data should fail')

assert failed_experiment.state.phase == TrainState.failed
assert failed_experiment.state.is_failed
assert failed_experiment.state.epoch == 0
assert failed_experiment.state.step == 0

## Managed persistence

Direct `TrainState` is an in-memory compatibility view, not persisted lifecycle authority. Durable training uses a completed `CachedDataset` input and a managed `Experiment.train` realization. The trained estimator is an immutable Store-backed output; saving the Experiment definition preserves the logical recipe, while `trained_model()` hydrates the active output before and after reopening the temporary Store.

In [ ]:
with TemporaryDirectory() as temporary_root:
    store_path = Path(temporary_root) / 'experiment-store'
    store = DirStore(store_path, query_index='memory')
    managed_data = CachedDataset(train_data)
    managed_data.compute(store=store, representation='numpy-sequence')
    managed_experiment = Experiment(
        RegressionModel(LinearRegression),
        BasicTraining(),
        train_data=managed_data,
    )
    train_result = managed_experiment.train(store=store)
    trained_model = managed_experiment.trained_model(store=store)
    expected_coef = trained_model.estimator.coef_.copy()
    expected_intercept = float(trained_model.estimator.intercept_)

    assert train_result.action == 'start'
    assert managed_experiment.train.status(store=store).status == 'completed'

    repo = Repo(stores=store)
    repo.save_definition(
        managed_experiment.definition, alias='linear-experiment'
    )
    repo.close(flush=True)

    reopened = Repo(stores=DirStore(store_path, query_index='memory'))
    try:
        restored = reopened.load_alias(
            'linear-experiment',
            instance='new',
            cache='none',
            restore_state=False,
        )
        restored_model = restored.trained_model(store=reopened.default_store)
        assert restored.definition == managed_experiment.definition
        assert restored.train.status(
            store=reopened.default_store
        ).status == 'completed'
        assert restored_model.obj is restored_model.estimator, 'estimator alias'
        assert hasattr(restored_model.estimator, 'coef_'), 'fitted coefficients'
        np.testing.assert_allclose(
            restored_model.estimator.coef_,
            expected_coef,
            err_msg='restored coefficients',
        )
        np.testing.assert_allclose(
            float(restored_model.estimator.intercept_),
            expected_intercept,
            err_msg='restored intercept',
        )
        np.testing.assert_allclose(
            restored_model(np.array([[3.0, 13.0]], dtype=np.float32)),
            np.array([4.0]),
            atol=1e-6,
            err_msg='restored prediction',
        )
    finally:
        reopened.close(flush=False)

## Further backend progression

This lesson uses the maintained sklearn implementation rather than a tutorial-owned model substitute. For later backend-specific work, consult the matching-version DRYML Models API documentation and the maintained `dryml.models.tf` APIs (including `Model`, `Sequential`, and `BasicTraining`) or `dryml.models.torch` APIs (including `Model`, `Sequential`, and `Optimizer`). TensorFlow and Torch have their own extras and runtime requirements; they are further progression, not prerequisites or substitutes for this tutorial.